# **TRAINING SCRIPT**
## *Chess Neural Network*

### **SETUP PHASE**

#### 1. Importing libraries

In [1]:
import torch
import pickle
import numpy as np

from tqdm import tqdm
from pathlib import Path
from model import ChessCNN
from torch.utils.data import DataLoader, TensorDataset

#### 2. Defining global variables

On essaie de forcer les calculs effectués lors de l'entrainement sur le GPU en priorité. Ensuite nous initialisons les variables d'entrainement :
- Une taille de batch à 64 - *le modèle ne voit que 64 positions par 64 positions*
- Un nombre d'epochs à 100 - *l'entrainement va faire 100 passages sur le dataset complet*
- Un learning rate (taux d'apprentissage) à 0.0001 - *à chaque batch le réseau se rend compte de ses erreurs et va légèrement corriger*  

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PHASE 1

LAMBDA_VALUE_P1 = 0.1
BATCH_SIZE = 64
EPOCHS_P1 = 50
LR_P1 = 1e-4

# PHASE 2 

EPOCHS_P2    = 20      
LR_P2        = 1e-5     

CKPT_DIR_P1  = Path("../Models/Checkpoints/Phase1")
CKPT_DIR_P2  = Path("../Models/Checkpoints/Phase2")
DATA_DIR     = Path("../Data/Processed Database")
MODEL_DIR    = Path("../Models")

for d in [CKPT_DIR_P1, CKPT_DIR_P2, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

### **PHASE 1 : GAMES**

#### 1. Loading datasets

On charge en mémoire les données préparées par le script `Scripts/dataset.py`.

- `X.npy` : Il prend la forme d'un tableau NumPy (N, 13, 8, 8) où chaque élément est une position d'échecs encodée
- `y.npy` : Il prend la forme d'un tableau NumPy (N) où chaque valeur valeur est l'indice d'un coup joué en partie (stocké dans X)

En sortant les données de cette manière nos faisons en sorte que le modèle ne prédit pas les coups 'texte' mais les indices des coups.

Enfin `num_moves` recense le nombre total de coups uniques dans le dataset et va définir la taille de la dernière couche du réseau.

In [ ]:
X_games = np.load(DATA_DIR / "X.npy")
y_games = np.load(DATA_DIR / "y.npy")

with open(DATA_DIR / "move_to_int.pkl", "rb") as f:
    move_to_int = pickle.load(f)

num_moves = len(move_to_int)

print(f"Positions (parties) : {len(y_games):,}")
print(f"Coups uniques       : {num_moves:,}")
print(f"Shape X             : {X_games.shape}")

dataset_games = TensorDataset(torch.tensor(X_games),
                              torch.tensor(y_games, dtype=torch.long))

#### 2. Loading model & optimizers

In [ ]:
loader = DataLoader(dataset_games, batch_size=BATCH_SIZE, 
                                   shuffle=True,
                                   num_workers=4,
                                   pin_memory="cuda")

model = ChessCNN(num_moves).to(DEVICE)

criterion_policy = torch.nn.CrossEntropyLoss()
criterion_value  = torch.nn.MSELoss()

optimizer_p1 = torch.optim.Adam(model.parameters(), lr=LR_P1)
scheduler_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_p1, mode='min', factor=0.5, patience=5, verbose=True)

#### 3. Traning model

On entraîne le modèle sur `EPOCHS` avec reprise automatique depuis un checkpoint (si disponible).
 
- **Reprise depuis un checkpoint**

Au démarrage, on cherche le checkpoint le plus récent dans `CKPT_DIR_P1`.

- **Boucle d'entraînement**

À chaque batch on effectue le cycle classique : *`zero_grad` → `forward` → `loss` → `backward` → `optimizer.step`*.

- **Perte**

La loss combine une '*policy loss*' `p_loss` pour prédire le coup et une '*value loss*' `v_loss` pour estimer la victoire du joueur courant.

- **Checkpoints & sauvegarde finale**

Un checkpoint est sauvegardé à la fin de chaque epoch. Il contient le modèle, l'optimizer et le numéro de la prochaine epoch (*epoch + 1*).

In [ ]:
CHECKPOINT_DIR = Path(CKPT_DIR_P1)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

existing = sorted(CHECKPOINT_DIR.glob("checkpoint_epoch_*.pth"))

if existing:

    latest = existing[-1]
    checkpoint = torch.load(latest, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state"])
    optimizer_p1.load_state_dict(checkpoint["optimizer_state"])

    start_epoch = checkpoint["epoch"]
    print(f"Reprise phase 1depuis {latest.name} (epoch {start_epoch})")

else:
    start_epoch = 0
    print("Phase 1 :Entraînement depuis zéro")

for epoch in range(start_epoch, EPOCHS_P1):

    model.train()

    total_loss   = 0.0
    total_p_loss = 0.0
    total_v_loss = 0.0
    correct      = 0
    total        = 0

    loop = tqdm(loader, desc=f"[P1] Epoch {epoch+1}/{EPOCHS_P1}", leave=True)

    for xb, yb in loop:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer_p1.zero_grad()
        logits, value = model(xb)
        p_loss = criterion_policy(logits, yb)

        turn_label = (xb[:, 12, 0, 0] * 2 - 1).unsqueeze(1)  
        v_loss = criterion_value(value, turn_label)

        loss = p_loss + LAMBDA_VALUE_P1 * v_loss
        loss.backward()

        optimizer_p1.step()

        total_loss   += loss.item()
        total_p_loss += p_loss.item()
        total_v_loss += v_loss.item()

        preds    = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total   += yb.size(0)

        loop.set_postfix(loss  = f"{loss.item():.4f}",
                         p_loss= f"{p_loss.item():.4f}",
                         v_loss= f"{v_loss.item():.4f}",
                         acc   = f"{100 * correct / total:.2f}%")

    avg_loss   = total_loss   / len(loader)
    avg_p_loss = total_p_loss / len(loader)
    avg_v_loss = total_v_loss / len(loader)
    accuracy   = 100 * correct / total

    print(f"[P1] Epoch {epoch+1}/{EPOCHS_P1} | "
          f"loss={avg_loss:.4f} | p_loss={avg_p_loss:.4f} | "
          f"v_loss={avg_v_loss:.4f} | acc={accuracy:.2f}%")

    scheduler_p1.step(avg_loss)

    # Checkpoint

    torch.save({"epoch"          : epoch + 1,
                "model_state"    : model.state_dict(),
                "optimizer_state": optimizer_p1.state_dict(),
                "num_moves"      : num_moves},

        CKPT_DIR_P1 / f"checkpoint_epoch_{epoch+1:03d}.pth")

# Sauvegarde Phase 1

MODEL_P1_PATH = MODEL_DIR / f"model_phase1_{EPOCHS_P1}ep.pth"

torch.save(model.state_dict(), MODEL_P1_PATH)
print(f"Phase 1 terminée — modèle sauvegardé : {MODEL_P1_PATH}")

### **PHASE 2 : PUZZLES**



On **gèle les couches convolutives** (le trunk) pour conserver les représentations apprises en Phase 1 et on n'entraîne que la **policy head** sur les coups de puzzles. La value head, elle, reste intacte en gardant ses poids de la Phase 1.

#### 1. Loading datasets

In [ ]:
X_puzzles = np.load(DATA_DIR / "X_puzzles.npy")
y_puzzles = np.load(DATA_DIR / "y_puzzles.npy")

with open(DATA_DIR / "move_to_int_puzzles.pkl", "rb") as f:
    move_to_int_puzzles = pickle.load(f)

num_moves_puzzles = len(move_to_int_puzzles)

print(f"Positions (puzzles) : {len(y_puzzles):,}")
print(f"Coups uniques       : {num_moves_puzzles:,}")
print(f"Shape X             : {X_puzzles.shape}")

#### 2. Loading new moves

In [ ]:
# --- Ajouts de nouveaux coups

# Coups présents dans les puzzles mais absents des parties
new_moves = {uci: idx for uci, idx in move_to_int_puzzles.items() if uci not in move_to_int}
print(f"Nouveaux coups (puzzles only) : {len(new_moves)}")

# On étend move_to_int avec les nouveaux coups
next_id = num_moves

for uci in new_moves:

    move_to_int[uci] = next_id
    next_id += 1

num_moves_total = len(move_to_int)
print(f"Vocabulaire total après fusion : {num_moves_total}")

# Réindexation de y_puzzles avec les nouveaux indices 
# On construit la table de correspondance old_puzzle_idx → new_global_idx

old_to_new = {old_idx: move_to_int[uci] for uci, old_idx in move_to_int_puzzles.items()}
y_puzzles_reindexed = np.vectorize(old_to_new.get)(y_puzzles).astype(np.int64)
print("Réindexation y_puzzles terminée")

# --- Extension du modèle

if num_moves_total > num_moves:

    old_weight = model.out_policy.weight.data  
    old_bias   = model.out_policy.bias.data    
    extra = num_moves_total - num_moves

    new_weight = torch.nn.init.xavier_uniform_(torch.empty(extra, old_weight.size(1))).to(DEVICE)
    new_bias = torch.zeros(extra).to(DEVICE)

    model.out_policy = torch.nn.Linear(256, num_moves_total).to(DEVICE)
    model.out_policy.weight.data = torch.cat([old_weight, new_weight], dim=0)
    model.out_policy.bias.data   = torch.cat([old_bias,   new_bias],   dim=0)

    print(f"Couche policy étendue : {num_moves} → {num_moves_total} sorties")
    
else:
    print("Aucune extension nécessaire")

# --- Chargement des données

dataset_puzzles = TensorDataset(torch.tensor(X_puzzles),
                                torch.tensor(y_puzzles_reindexed, dtype=torch.long))

loader_puzzles = DataLoader(dataset_puzzles, batch_size=BATCH_SIZE,
                                             shuffle=True,
                                             num_workers=2,
                                             pin_memory=(DEVICE.type == "cuda"))

#### 3. Model freezing & optimizer loading

In [ ]:
# --- Gel des premières couches

# On gèle conv1, conv2, conv3 7
# Le tronc reste fixe

frozen_layers = ["conv1", "conv2", "conv3"]

for name, param in model.named_parameters():
    layer = name.split(".")[0]

    if layer in frozen_layers:
        param.requires_grad = False

print("Paramètres entraînables en Phase 2 :")

for name, param in model.named_parameters():

    if param.requires_grad:
        print(f"  V {name}")

    else:
        print(f"  X {name} (gelé)")

# -- Chargement du checkpoint Phase 2 

# Avant on optmise les paramètres non gelés
optimizer_p2 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_P2)

existing_p2 = sorted(CKPT_DIR_P2.glob("checkpoint_epoch_*.pth"))

if existing_p2:
    
    latest = existing_p2[-1]
    ckpt   = torch.load(latest, map_location=DEVICE)

    model.load_state_dict(ckpt["model_state"])
    optimizer_p2.load_state_dict(ckpt["optimizer_state"])
    start_epoch_p2 = ckpt["epoch"]

    print(f"Reprise Phase 2 depuis {latest.name} (epoch {start_epoch_p2})")
else:
    start_epoch_p2 = 0
    print("Phase 2 : fine-tuning depuis le modèle Phase 1")

#### 4. Model training (fine-tuning)

In [ ]:
for epoch in range(start_epoch_p2, EPOCHS_P2):

    model.train()

    total_loss = 0.0
    correct    = 0
    total      = 0

    loop = tqdm(loader_puzzles, desc=f"[P2] Epoch {epoch+1}/{EPOCHS_P2}", leave=True)

    for xb, yb in loop:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer_p2.zero_grad()

        logits, _ = model(xb)   

        loss = criterion_policy(logits, yb)
        loss.backward()

        optimizer_p2.step()

        total_loss += loss.item()
        preds       = logits.argmax(dim=1)
        correct    += (preds == yb).sum().item()
        total      += yb.size(0)

        loop.set_postfix(loss = f"{loss.item():.4f}",
                         acc  = f"{100 * correct / total:.2f}%")

    avg_loss = total_loss / len(loader_puzzles)
    accuracy = 100 * correct / total

    print(f"[P2] Epoch {epoch+1}/{EPOCHS_P2} | "
          f"loss={avg_loss:.4f} | acc={accuracy:.2f}%")

    # Checkpoint

    torch.save({"epoch"          : epoch + 1,
                "model_state"    : model.state_dict(),
                "optimizer_state": optimizer_p2.state_dict(),
                "num_moves"      : num_moves_total},

        CKPT_DIR_P2 / f"checkpoint_epoch_{epoch+1:03d}.pth")

# Sauvegarde finale Phase 2

MODEL_P2_PATH = MODEL_DIR / f"model_phase2_{EPOCHS_P1}+{EPOCHS_P2}ep.pth"
torch.save(model.state_dict(), MODEL_P2_PATH)

print(f"Phase 2 terminée — modèle sauvegardé : {MODEL_P2_PATH}")

#### 5. Saving Vocabulary

In [ ]:
FINAL_VOCAB_PATH = DATA_DIR / "move_to_int_final.pkl"

with open(FINAL_VOCAB_PATH, "wb") as f:
    pickle.dump(move_to_int, f)

print(f"Vocabulaire final sauvegardé : {FINAL_VOCAB_PATH}")
print(f"Total coups connus           : {num_moves_total}")